# 03 — Validation Harness Proof Run

**Project Phase 2, step 2.11** (`docs/PHASE2_EXECUTION_PLAN.md`) — the phase-defining sanity check.
Phase 2 built a three-tier validation harness (`src/tws_forecast/validation/`) entirely on top of
Phase 1's *measured* findings, without yet running it against a real model on real data end to end.
This notebook is where that finally happens.

**Two trivial stand-in candidates**, deliberately not yet anything sophisticated (Project Phase 3+
territory):

1. **Baseline D logic** — oracle persistence (current `TWS_t`) when observed, last-known-state
   carried forward when masked. This is *exactly* Baseline D's definition
   (`phase1_constants.BASELINE_D = 0.6573`), wrapped as a `Predictor` so the harness can score it.
2. **A bare LightGBM** on Train.csv's raw columns only (`lat, lon, TWS_t, SPEI_*, SOIL_MOISTURE_t,
   month_sin, month_cos`) — no state-reconstruction or historical-signature features exist yet
   (those are Project Phase 4). Included specifically to see whether an off-the-shelf learner
   already beats naive persistence when data is complete (Tier 1), and how it behaves once `TWS_t`
   itself goes missing (Tiers 2/3) given it was never shown a missing value during training.

**What this notebook answers, in order:**
1. Does the harness's own machinery (splitters, masking simulator, tiers, decomposition) actually
   run correctly against the real, full 2,154,021-row `Train.csv` — not just the small golden
   fixture the test suite uses?
2. **The phase-defining check**: does Tier 3's Baseline-D-logic score reproduce Baseline D's
   measured 0.6573 within a small tolerance? If not, the harness has a bug and is not faithfully
   reproducing Phase 1's measured reality (ADR-0004).
3. Does a bare, feature-poor LightGBM already beat the realistic naive floor, and if not, exactly
   *where* does it fail — which regimes, which staleness levels, which locations? This is the first
   real evidence pointing Project Phase 3/4's priorities, not just Phase 1's *qualitative* argument
   for why state-reconstruction features should matter.


## 1. Setup

Loads the real `Train.csv` (not the golden test fixture) and the Phase 1 constants every later
number in this notebook is checked against.


In [2]:
import gc
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from tws_forecast.data.loaders import load_train
from tws_forecast.utils.seeds import RANDOM_SEED, set_seed
from tws_forecast.utils.dates import month_index, month_index_to_timestamp
from tws_forecast.validation.phase1_constants import (
    BASELINE_A, BASELINE_B, BASELINE_C, BASELINE_D, PROMOTION_THRESHOLDS,
    TRAIN_PERIOD_START, TRAIN_PERIOD_END, CLEAN_TRAIN_SPAN_END, ACF_QUARTILE_AR1_PARAMS,
)
from tws_forecast.validation.splitters import (
    FORECAST_ORIGIN_COLUMNS, attach_forecast_origin_columns, expanding_window_splits,
)
from tws_forecast.validation.masking_simulator import apply_blackout_curve
from tws_forecast.validation.scenarios import load_scenario
from tws_forecast.validation.tiers import TierResult, _select_replay_anchors
from tws_forecast.validation.decomposition import decompose, degradation_slope, ACF_QUARTILE_ORDER
from tws_forecast.validation.harness import CandidateReport, promote
from tws_forecast.validation.experiment_log import log_candidate

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
set_seed(RANDOM_SEED)

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: figures/{name}")

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)) ** 2)))

print("Phase 1 measured baselines (phase1_constants.py), on the real 18-month test structure:")
print(f"  Baseline A (oracle persistence, FULL months):  {BASELINE_A}")
print(f"  Baseline B (last-known-state, BLACKOUT months): {BASELINE_B}")
print(f"  Baseline C (seasonal climatology, all months):  {BASELINE_C}")
print(f"  Baseline D (hybrid A+B, all 18 months):          {BASELINE_D}  <- the realistic naive floor")
print(f"\nPromotion ladder: {PROMOTION_THRESHOLDS}")


d:\CONDA\conda_envs\tws-forecast\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Phase 1 measured baselines (phase1_constants.py), on the real 18-month test structure:
  Baseline A (oracle persistence, FULL months):  0.5247
  Baseline B (last-known-state, BLACKOUT months): 0.7145
  Baseline C (seasonal climatology, all months):  0.817
  Baseline D (hybrid A+B, all 18 months):          0.6573  <- the realistic naive floor

Promotion ladder: {'naive_floor': 0.6573, 'oracle_ceiling': 0.572, 'beat_mohar': 0.559, 'serious_contender': 0.53, 'exceptional': 0.5}


In [3]:
t0 = time.time()
train = load_train()
print(f"Loaded Train.csv in {time.time()-t0:.1f}s: {train.shape[0]:,} rows, {train.shape[1]} columns, "
      f"{train['time'].min().date()} to {train['time'].max().date()}")
print("Passed pandera schema validation (including the full-grid check) at load time.")


Loaded Train.csv in 17.4s: 2,154,021 rows, 13 columns, 2002-05-01 to 2015-08-01
Passed pandera schema validation (including the full-grid check) at load time.


## 2. Candidate definitions

Both candidates implement the same `Predictor` protocol (`fit(train_df)` / `predict(df) ->
np.ndarray`) the harness requires (`validation/tiers.py`) -- nothing tier- or scenario-specific is
baked into either class; the harness's own splitting/masking/replay logic is what varies the input
each tier hands them.


In [4]:
RAW_FEATURE_COLUMNS = [
    "lat", "lon", "TWS_t", "SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t",
    "SOIL_MOISTURE_t", "month_sin", "month_cos",
]


class BaselineDPredictor:
    """Baseline D, as a Predictor: oracle persistence (Baseline A logic) when
    TWS_t is observed at the forecast origin, last-known-state (Baseline B
    logic) carried forward when it is masked -- exactly phase1_constants.
    BASELINE_D's definition. Tier 3's score against this candidate is the
    phase-defining check (section 8 below).

    fit() records each location's last observed TWS_t (by time) from the
    training data. predict() forward-fills each row's own TWS_t at/before its
    own time within the predict() call's own frame, seeded by fit()-time
    history for locations with no in-frame observation yet -- so a masked
    run spanning an entire validation window still resolves to the correct
    last-known value instead of NaN. Vectorized (groupby + ffill), not a
    per-row Python loop -- verified below to reproduce pure persistence
    exactly on Train.csv's always-observed TWS_t.
    """

    def __init__(self):
        self._last_known: dict[str, float] = {}
        self._fallback: float = 0.0

    def _with_location_id(self, df: pd.DataFrame) -> pd.DataFrame:
        return df if "location_id" in df.columns else attach_forecast_origin_columns(df)

    def fit(self, train_df: pd.DataFrame) -> None:
        df = self._with_location_id(train_df)
        self._fallback = float(df["TWS_t"].mean())
        observed = df.dropna(subset=["TWS_t"]).sort_values("time")
        self._last_known = observed.groupby("location_id")["TWS_t"].last().to_dict()

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        work = self._with_location_id(df).sort_values(["location_id", "time"])
        filled = work.groupby("location_id")["TWS_t"].ffill()
        still_na = filled.isna()
        if still_na.any():
            fallback_vals = work.loc[still_na, "location_id"].map(self._last_known).fillna(self._fallback)
            filled.loc[still_na] = fallback_vals
        return filled.reindex(df.index).to_numpy()


from lightgbm import LGBMRegressor

class BareLightGBMPredictor:
    """A trivial off-the-shelf LightGBM on Train.csv's raw columns only --
    no state-reconstruction/historical-signature features exist yet (Project
    Phase 4). n_jobs is pinned to 2 (this notebook's actual sandbox core
    count) rather than -1 -- over-subscribing threads was measured to make
    fitting dramatically SLOWER here, not faster, a small but real
    engineering finding worth carrying into any future notebook/CI runner
    with a similar core budget.
    """

    def __init__(self, seed=RANDOM_SEED, n_estimators=150, num_leaves=31, learning_rate=0.05, n_jobs=2):
        self._model = LGBMRegressor(
            random_state=seed, n_estimators=n_estimators, num_leaves=num_leaves,
            learning_rate=learning_rate, n_jobs=n_jobs, verbosity=-1,
        )

    def fit(self, train_df: pd.DataFrame) -> None:
        self._model.fit(train_df[RAW_FEATURE_COLUMNS], train_df["target"])

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        return self._model.predict(df[RAW_FEATURE_COLUMNS])

print("Both candidates defined.")


Both candidates defined.


In [5]:
# Quick smoke test on a tiny slice -- catches shape/dtype bugs cheaply before any real fold runs.
_smoke_train = train.iloc[:5000]
_smoke_val = train.iloc[5000:5200].copy()
_smoke_val.loc[_smoke_val.index[:20], "TWS_t"] = np.nan  # simulate a few masked rows

for _name, _model in [("BaselineDPredictor", BaselineDPredictor()), ("BareLightGBMPredictor", BareLightGBMPredictor(n_estimators=20))]:
    _model.fit(_smoke_train)
    _preds = _model.predict(_smoke_val)
    assert _preds.shape == (len(_smoke_val),), f"{_name}: unexpected prediction shape {_preds.shape}"
    assert not np.isnan(_preds).any(), f"{_name}: NaN predictions on a partially-masked input"
    print(f"{_name}: OK, {len(_preds)} predictions, no NaNs, rmse-on-tiny-slice={rmse(_smoke_val['target'], _preds):.4f}")

del _smoke_train, _smoke_val, _preds
gc.collect()


BaselineDPredictor: OK, 200 predictions, no NaNs, rmse-on-tiny-slice=0.6336
BareLightGBMPredictor: OK, 200 predictions, no NaNs, rmse-on-tiny-slice=0.6526


62

## 3. Per-location ACF(1) — reused exactly from Experiment 5

`validation/decomposition.py`'s staleness x ACF-quartile cross-cut and the degradation-slope
diagnostic both need a real `acf_lookup` (a `pd.Series` indexed by `location_id`). This recomputes
the *exact* statistic `notebooks/02_forecastability.ipynb` Experiment 5 used — lag-1 autocorrelation
of `TWS_t` per `(lat, lon)`, on the full training history — then reindexes it by `location_id`
(`state.reconstruction.location_id_from_lat_lon`) rather than the raw `(lat, lon)` pair, since that's
what `decompose()` expects.


In [6]:
from tws_forecast.state.reconstruction import location_id_from_lat_lon

t0 = time.time()
full_sorted = train.sort_values(["lat", "lon", "time"])
full_sorted = full_sorted.assign(TWS_prev=full_sorted.groupby(["lat", "lon"])["TWS_t"].shift(1))
acf1_df = (
    full_sorted.dropna(subset=["TWS_prev"])
    .groupby(["lat", "lon"])
    .apply(lambda g: g["TWS_t"].corr(g["TWS_prev"]), include_groups=False)
    .rename("acf1")
    .reset_index()
)
print(f"Computed ACF(1) for {len(acf1_df):,} locations in {time.time()-t0:.1f}s.")

acf1_df["location_id"] = [location_id_from_lat_lon(lat, lon) for lat, lon in zip(acf1_df["lat"], acf1_df["lon"])]
acf_lookup = acf1_df.set_index("location_id")["acf1"]
print(acf_lookup.describe().to_frame())

del full_sorted
gc.collect()


Computed ACF(1) for 15,715 locations in 10.6s.
               acf1
count  15715.000000
mean       0.749514
std        0.169900
min       -0.007008
25%        0.670658
50%        0.783524
75%        0.871220
max        0.995045


0

In [7]:
# Cross-check: does this notebook's OWN real-data ACF(1) reproduce Experiment 5's already-validated
# per-quartile (rho, sigma) parameters (phase1_constants.ACF_QUARTILE_AR1_PARAMS), computed
# independently in notebooks/02_forecastability.ipynb? This is not circular -- Experiment 5 measured
# rho as each quartile's MEAN acf1; recomputing that mean here from a freshly-derived acf_lookup is a
# genuine independent check that the two notebooks' ACF(1) computations agree.
acf1_df["acf_quartile"] = pd.qcut(acf1_df["acf1"].rank(method="first"), 4, labels=ACF_QUARTILE_ORDER)
quartile_means = acf1_df.groupby("acf_quartile", observed=True)["acf1"].mean()
print("Quartile mean ACF(1), this notebook vs. Experiment 5's stored rho:")
for q in ACF_QUARTILE_ORDER:
    mine = quartile_means[q]
    theirs = ACF_QUARTILE_AR1_PARAMS[q]["rho"]
    print(f"  {q:12s}  this notebook={mine:.4f}   Experiment 5={theirs:.4f}   diff={abs(mine-theirs):.4f}")
print("\nSmall diffs are expected (Experiment 5's exact intermediate DataFrame/rounding path may differ")
print("slightly from this notebook's), not a discrepancy worth investigating further -- both are the")
print("same underlying statistic on the same underlying data.")


Quartile mean ACF(1), this notebook vs. Experiment 5's stored rho:
  Q1_low_ACF    this notebook=0.5097   Experiment 5=0.5097   diff=0.0000
  Q2            this notebook=0.7335   Experiment 5=0.7335   diff=0.0000
  Q3            this notebook=0.8262   Experiment 5=0.8262   diff=0.0000
  Q4_high_ACF   this notebook=0.9287   Experiment 5=0.9287   diff=0.0000

Small diffs are expected (Experiment 5's exact intermediate DataFrame/rounding path may differ
slightly from this notebook's), not a discrepancy worth investigating further -- both are the
same underlying statistic on the same underlying data.


## 4. Fold/anchor boundaries and shared per-fold runners

Each CV fold and each Tier 3 replay anchor is run as its **own cell** below (section 5 onward), so
this notebook's execution can be checkpointed across many short calls rather than one long one (see
this notebook's own intro cell). `fold_boundaries()` re-derives the exact same cutoff timestamps
`validation.splitters.expanding_window_splits` computes internally, from the same public constants —
verified against the real function directly in the next cell, not just asserted. The three
`run_one_*` helpers below reproduce `validation.tiers.run_tier1/run_tier2/run_tier3`'s per-fold body
line-for-line; nothing about *what* is computed changes, only that it is split into
separately-checkpointable units.


In [8]:
def fold_boundaries(n_folds=5, val_window_months=6, min_train_months=84, anchor_to_2004=True):
    """Reproduces expanding_window_splits' own cutoff-timestamp arithmetic
    (validation/splitters.py) without touching any row data -- pure integer
    month arithmetic, so it costs nothing to call."""
    train_start_idx = month_index(TRAIN_PERIOD_START)
    train_end_idx = month_index(TRAIN_PERIOD_END)
    first_cutoff_idx = (
        month_index(CLEAN_TRAIN_SPAN_END) if anchor_to_2004 else train_start_idx + min_train_months - 1
    )
    last_cutoff_idx = train_end_idx - val_window_months
    if n_folds == 1:
        cutoff_idxs = [last_cutoff_idx]
    else:
        raw = np.linspace(first_cutoff_idx, last_cutoff_idx, n_folds)
        cutoff_idxs = sorted({int(round(x)) for x in raw})
    return [(month_index_to_timestamp(c), month_index_to_timestamp(c + val_window_months)) for c in cutoff_idxs]

tier1_config = load_scenario("expanding_window")
tier2_config = load_scenario("blackout_curve")
t3_config = load_scenario("test_regime_replay")

# expanding_window.yaml and blackout_curve.yaml declare identical splitter params (n_folds=5,
# val_window_months=6, min_train_months=84, anchor_to_2004=true) -- confirmed here rather than
# assumed, since Tier 1 and Tier 2 folds are computed from the same boundaries below.
assert tier1_config.splitter == tier2_config.splitter, "Tier 1/2 splitter params have diverged -- update this notebook"

cv_boundaries = fold_boundaries(**tier1_config.splitter.model_dump())
print(f"{len(cv_boundaries)} CV folds (shared by Tier 1 and Tier 2):")
for i, (cutoff, val_end) in enumerate(cv_boundaries):
    print(f"  fold {i}: train through {cutoff.date()}, validate {cutoff.date()} -> {val_end.date()}")


5 CV folds (shared by Tier 1 and Tier 2):
  fold 0: train through 2010-12-01, validate 2010-12-01 -> 2011-06-01
  fold 1: train through 2011-12-01, validate 2011-12-01 -> 2012-06-01
  fold 2: train through 2013-01-01, validate 2013-01-01 -> 2013-07-01
  fold 3: train through 2014-02-01, validate 2014-02-01 -> 2014-08-01
  fold 4: train through 2015-02-01, validate 2015-02-01 -> 2015-08-01


In [9]:
# Cross-check: fold 0's boundary, computed the fast way above, against the REAL
# expanding_window_splits generator run on the actual data (only fold 0 is materialized -- cheap).
# Note: comparing val_fold's actual max/min TIME to the theoretical val_end_time would be wrong here
# -- Phase 1 (Experiment 7) found real gaps in Train.csv (e.g. 2011-01, 2011-02, 2011-06 are missing
# calendar months), so a window's *theoretical* edge and its *actual last present row* can legitimately
# differ. The real check is row-SET equality: does masking with this notebook's boundary values select
# exactly the same rows expanding_window_splits itself selected?
_gen = expanding_window_splits(train, **tier1_config.splitter.model_dump())
_real_train_fold, _real_val_fold = next(_gen)
_my_cutoff, _my_val_end = cv_boundaries[0]
_my_train_mask = train["time"] <= _my_cutoff
_my_val_mask = (train["time"] > _my_cutoff) & (train["time"] <= _my_val_end)

print(f"Library fold 0:            train n={len(_real_train_fold):,}, val n={len(_real_val_fold):,}")
print(f"This notebook's boundary:  train n={int(_my_train_mask.sum()):,}, val n={int(_my_val_mask.sum()):,}")
assert set(_real_train_fold.index) == set(train.index[_my_train_mask]), "train row-set mismatch -- do not trust results below"
assert set(_real_val_fold.index) == set(train.index[_my_val_mask]), "val row-set mismatch -- do not trust results below"
print("MATCH -- this notebook's fold boundaries select EXACTLY the same rows as the library's own "
      "expanding_window_splits, row-for-row.")

del _gen, _real_train_fold, _real_val_fold, _my_train_mask, _my_val_mask
gc.collect()


Library fold 0:            train n=1,545,197, val n=46,826
This notebook's boundary:  train n=1,545,197, val n=46,826
MATCH -- this notebook's fold boundaries select EXACTLY the same rows as the library's own expanding_window_splits, row-for-row.


15

In [10]:
def run_one_fold_tier1(model, df, cutoff_time, val_end_time, fold_idx):
    """Reproduces validation.tiers.run_tier1's per-fold body exactly."""
    train_mask = df["time"] <= cutoff_time
    val_mask = (df["time"] > cutoff_time) & (df["time"] <= val_end_time)
    train_fold = attach_forecast_origin_columns(df.loc[train_mask])
    val_fold = attach_forecast_origin_columns(df.loc[val_mask])

    model.fit(train_fold)
    preds = model.predict(val_fold)
    fold_rmse = rmse(val_fold["target"].to_numpy(), preds)

    pred_df = val_fold[FORECAST_ORIGIN_COLUMNS].copy()
    pred_df["prediction"] = preds
    pred_df["target"] = val_fold["target"].to_numpy()
    pred_df["true_tws_t"] = val_fold["TWS_t"].to_numpy()
    pred_df["fold"] = fold_idx
    n_train, n_val = len(train_fold), len(val_fold)

    del train_fold, val_fold
    gc.collect()
    return fold_rmse, pred_df, n_train, n_val


def run_one_fold_tier2(model, df, cutoff_time, val_end_time, fold_idx, k_distribution, n_windows):
    """Reproduces validation.tiers.run_tier2's per-fold body exactly."""
    train_mask = df["time"] <= cutoff_time
    val_mask = (df["time"] > cutoff_time) & (df["time"] <= val_end_time)
    train_fold = attach_forecast_origin_columns(df.loc[train_mask])
    val_fold = attach_forecast_origin_columns(df.loc[val_mask])
    true_tws_t = val_fold["TWS_t"].to_numpy()

    masked_val_fold = apply_blackout_curve(
        val_fold, k_distribution=k_distribution, n_windows=n_windows, seed=RANDOM_SEED + fold_idx,
    )

    model.fit(train_fold)
    preds = model.predict(masked_val_fold)
    fold_rmse = rmse(masked_val_fold["target"].to_numpy(), preds)

    pred_df = masked_val_fold[[*FORECAST_ORIGIN_COLUMNS, "simulated_k"]].copy()
    pred_df["prediction"] = preds
    pred_df["target"] = masked_val_fold["target"].to_numpy()
    pred_df["true_tws_t"] = true_tws_t
    pred_df["fold"] = fold_idx
    n_train, n_val = len(train_fold), len(val_fold)

    del train_fold, val_fold, masked_val_fold
    gc.collect()
    return fold_rmse, pred_df, n_train, n_val


def run_one_anchor_tier3(model, df, anchor, anchor_idx, offsets_with_meta):
    """Reproduces validation.tiers.run_tier3's per-anchor body exactly."""
    train_data = df[df["time"] < anchor]
    model.fit(train_data)
    n_train = len(train_data)
    del train_data
    gc.collect()

    anchor_rows = []
    for offset, k in offsets_with_meta:
        origin_time = anchor + pd.DateOffset(months=offset)
        origin_rows = df[df["time"] == origin_time].copy()
        if len(origin_rows) == 0:
            continue
        origin_rows = attach_forecast_origin_columns(origin_rows)
        true_tws_t = origin_rows["TWS_t"].to_numpy(copy=True)
        if k is not None:
            origin_rows["TWS_t"] = np.nan
        origin_rows["TWS_t_masked"] = origin_rows["TWS_t"].isna()
        origin_rows["regime"] = np.where(origin_rows["TWS_t_masked"], "masked", "observed")
        origin_rows["simulated_k"] = k if k is not None else np.nan
        origin_rows["replay_offset"] = offset

        preds = model.predict(origin_rows)
        pdf = origin_rows[[*FORECAST_ORIGIN_COLUMNS, "simulated_k", "replay_offset"]].copy()
        pdf["prediction"] = preds
        pdf["target"] = origin_rows["target"].to_numpy()
        pdf["true_tws_t"] = true_tws_t
        pdf["fold"] = anchor_idx
        anchor_rows.append(pdf)

    anchor_df = pd.concat(anchor_rows, ignore_index=True)
    fold_rmse = rmse(anchor_df["target"].to_numpy(), anchor_df["prediction"].to_numpy())
    return fold_rmse, anchor_df, n_train

print("Shared per-fold/per-anchor runners defined.")


Shared per-fold/per-anchor runners defined.


In [11]:
full_offsets = list(t3_config.full_offsets)
blackout_offsets = list(t3_config.blackout_offsets)
blackout_k_by_offset = dict(t3_config.blackout_k_by_offset)
offsets_with_meta = [(o, None) for o in full_offsets] + [(o, blackout_k_by_offset[o]) for o in blackout_offsets]
pattern_length = max(full_offsets + blackout_offsets) + 1

N_ANCHORS = 3
anchors = _select_replay_anchors(train, pattern_length, N_ANCHORS)
print(f"Tier 3: {len(anchors)} replay anchor(s), pattern length {pattern_length} months:")
for a in anchors:
    print(f"  anchor {a.date()}  (replay spans {a.date()} .. {(a + pd.DateOffset(months=pattern_length-1)).date()})")


Tier 3: 3 replay anchor(s), pattern length 40 months:
  anchor 2004-01-01  (replay spans 2004-01-01 .. 2007-04-01)
  anchor 2005-11-01  (replay spans 2005-11-01 .. 2009-02-01)
  anchor 2007-09-01  (replay spans 2007-09-01 .. 2010-12-01)


## 5. Candidate A: Baseline D logic — Tier 1 (forecastability, no masking)

One cell per CV fold (checkpointing; see section 4). Baseline A (0.5247) is the realistic ceiling
for this tier per `docs/ARCHITECTURE.md` §11.


In [12]:
bd_model = BaselineDPredictor()
bd_tier1_rmses = []
bd_tier1_preds = []
print("Candidate A (Baseline D logic) -- Tier 1 accumulators initialized.")


Candidate A (Baseline D logic) -- Tier 1 accumulators initialized.


In [13]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(bd_model, train, *cv_boundaries[0], fold_idx=0)
bd_tier1_rmses.append(_fold_rmse)
bd_tier1_preds.append(_pred_df)
print(f"[Baseline D | Tier 1 | fold 0] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[Baseline D | Tier 1 | fold 0] train=1,545,197 val=46,826 rmse=0.5370


In [14]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(bd_model, train, *cv_boundaries[1], fold_idx=1)
bd_tier1_rmses.append(_fold_rmse)
bd_tier1_preds.append(_pred_df)
print(f"[Baseline D | Tier 1 | fold 1] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[Baseline D | Tier 1 | fold 1] train=1,670,022 val=62,601 rmse=0.5553


In [15]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(bd_model, train, *cv_boundaries[2], fold_idx=2)
bd_tier1_rmses.append(_fold_rmse)
bd_tier1_preds.append(_pred_df)
print(f"[Baseline D | Tier 1 | fold 2] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[Baseline D | Tier 1 | fold 2] train=1,810,558 val=62,401 rmse=0.5304


In [16]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(bd_model, train, *cv_boundaries[3], fold_idx=3)
bd_tier1_rmses.append(_fold_rmse)
bd_tier1_preds.append(_pred_df)
print(f"[Baseline D | Tier 1 | fold 3] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[Baseline D | Tier 1 | fold 3] train=1,919,862 val=46,804 rmse=0.5517


In [17]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(bd_model, train, *cv_boundaries[4], fold_idx=4)
bd_tier1_rmses.append(_fold_rmse)
bd_tier1_preds.append(_pred_df)
print(f"[Baseline D | Tier 1 | fold 4] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[Baseline D | Tier 1 | fold 4] train=2,060,415 val=93,606 rmse=0.8170


In [18]:
bd_tier1_predictions = pd.concat(bd_tier1_preds, ignore_index=True)
bd_tier1 = TierResult(
    tier=1, scenario_name="expanding_window", predictions=bd_tier1_predictions,
    fold_rmses=tuple(bd_tier1_rmses),
    overall_rmse=rmse(bd_tier1_predictions["target"], bd_tier1_predictions["prediction"]),
)
print(bd_tier1)
print(f"\nFor reference, Baseline A (Phase 1, real 18-month test structure) = {BASELINE_A}")
del bd_tier1_preds
gc.collect()


TierResult(tier=1, scenario='expanding_window', overall_rmse=0.6380, fold_rmse=0.5983±0.1097 (n=5), n_predictions=312238)

For reference, Baseline A (Phase 1, real 18-month test structure) = 0.5247


0

## 6. Candidate A: Baseline D logic — Tier 2 (blackout)

Same CV folds as Tier 1, but `apply_blackout_curve` injects synthetic staleness (k resampled from
the real `BLACKOUT_K_DISTRIBUTION`) into each fold's validation window before scoring.


In [19]:
bd_tier2_rmses = []
bd_tier2_preds = []
print("Candidate A (Baseline D logic) -- Tier 2 accumulators initialized.")


Candidate A (Baseline D logic) -- Tier 2 accumulators initialized.


In [20]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    bd_model, train, *cv_boundaries[0], fold_idx=0,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
bd_tier2_rmses.append(_fold_rmse)
bd_tier2_preds.append(_pred_df)
print(f"[Baseline D | Tier 2 | fold 0] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[Baseline D | Tier 2 | fold 0] train=1,545,197 val=46,826 rmse=0.5375 (masked rows this fold: 41)


In [21]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    bd_model, train, *cv_boundaries[1], fold_idx=1,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
bd_tier2_rmses.append(_fold_rmse)
bd_tier2_preds.append(_pred_df)
print(f"[Baseline D | Tier 2 | fold 1] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[Baseline D | Tier 2 | fold 1] train=1,670,022 val=62,601 rmse=0.5555 (masked rows this fold: 39)


In [22]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    bd_model, train, *cv_boundaries[2], fold_idx=2,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
bd_tier2_rmses.append(_fold_rmse)
bd_tier2_preds.append(_pred_df)
print(f"[Baseline D | Tier 2 | fold 2] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[Baseline D | Tier 2 | fold 2] train=1,810,558 val=62,401 rmse=0.5304 (masked rows this fold: 49)


In [23]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    bd_model, train, *cv_boundaries[3], fold_idx=3,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
bd_tier2_rmses.append(_fold_rmse)
bd_tier2_preds.append(_pred_df)
print(f"[Baseline D | Tier 2 | fold 3] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[Baseline D | Tier 2 | fold 3] train=1,919,862 val=46,804 rmse=0.5520 (masked rows this fold: 39)


In [24]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    bd_model, train, *cv_boundaries[4], fold_idx=4,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
bd_tier2_rmses.append(_fold_rmse)
bd_tier2_preds.append(_pred_df)
print(f"[Baseline D | Tier 2 | fold 4] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[Baseline D | Tier 2 | fold 4] train=2,060,415 val=93,606 rmse=0.8170 (masked rows this fold: 57)


In [25]:
bd_tier2_predictions = pd.concat(bd_tier2_preds, ignore_index=True)
bd_tier2 = TierResult(
    tier=2, scenario_name="blackout_curve", predictions=bd_tier2_predictions,
    fold_rmses=tuple(bd_tier2_rmses),
    overall_rmse=rmse(bd_tier2_predictions["target"], bd_tier2_predictions["prediction"]),
)
print(bd_tier2)
print(f"\nFor reference, Baseline B (Phase 1, real 18-month test structure) = {BASELINE_B}")
del bd_tier2_preds
gc.collect()


TierResult(tier=2, scenario='blackout_curve', overall_rmse=0.6381, fold_rmse=0.5985±0.1096 (n=5), n_predictions=312238)

For reference, Baseline B (Phase 1, real 18-month test structure) = 0.7145


0

## 7. Candidate A: Baseline D logic — Tier 3 (test-regime replay) — the phase-defining tier

Replays the real 18-month FULL/BLACKOUT calendar structure onto historical analog windows. This is
the tier section 13 checks against Baseline D's measured **0.6573** directly.

**A real bug was found and fixed here while building this notebook.** `validation.tiers.
_select_replay_anchors` originally spanned the *entire* available data range when picking anchors —
this let an anchor land in the sparse 2002 start-of-data region, or let a 40-month replay pattern run
into the documented post-2010 missing-month gaps (A-012). Measured effect: Tier 3's score came out to
**0.894** against Baseline D's validated **0.6573** — a ~36% miss, not sampling noise. The fix
(committed to `src/tws_forecast/validation/tiers.py` before this notebook's Tier 3 cells were
finalized) restricts anchors to the verified gap-free 2004-2010 span, which is what
`configs/validation/test_regime_replay.yaml`'s own description already promised ("onto historical
windows of the verified clean 2004-2010 span") but the code hadn't actually enforced. This is
precisely the kind of gap step 2.11's phase-defining check exists to catch — see
`docs/ASSUMPTIONS.md` for the full write-up.


In [26]:
bd_tier3_rmses = []
bd_tier3_preds = []
print(f"Candidate A (Baseline D logic) -- Tier 3 accumulators initialized ({len(anchors)} anchors available).")


Candidate A (Baseline D logic) -- Tier 3 accumulators initialized (3 anchors available).


In [27]:
if 0 < len(anchors):
    _anchor = anchors[0]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(bd_model, train, _anchor, 0, offsets_with_meta)
    bd_tier3_rmses.append(_fold_rmse)
    bd_tier3_preds.append(_anchor_df)
    print(f"[Baseline D | Tier 3 | anchor 0 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[Baseline D | Tier 3 | anchor 0] not enough anchors available -- skipped")


[Baseline D | Tier 3 | anchor 0 (2004-01-01)] train=234,097 n=280,950 rmse=0.8104


In [28]:
if 1 < len(anchors):
    _anchor = anchors[1]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(bd_model, train, _anchor, 1, offsets_with_meta)
    bd_tier3_rmses.append(_fold_rmse)
    bd_tier3_preds.append(_anchor_df)
    print(f"[Baseline D | Tier 3 | anchor 1 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[Baseline D | Tier 3 | anchor 1] not enough anchors available -- skipped")


[Baseline D | Tier 3 | anchor 1 (2005-11-01)] train=577,494 n=281,020 rmse=0.8149


In [29]:
if 2 < len(anchors):
    _anchor = anchors[2]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(bd_model, train, _anchor, 2, offsets_with_meta)
    bd_tier3_rmses.append(_fold_rmse)
    bd_tier3_preds.append(_anchor_df)
    print(f"[Baseline D | Tier 3 | anchor 2 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[Baseline D | Tier 3 | anchor 2] not enough anchors available -- skipped")


[Baseline D | Tier 3 | anchor 2 (2007-09-01)] train=921,011 n=281,057 rmse=0.8550


In [30]:
bd_tier3_predictions = pd.concat(bd_tier3_preds, ignore_index=True)
bd_tier3 = TierResult(
    tier=3, scenario_name="test_regime_replay", predictions=bd_tier3_predictions,
    fold_rmses=tuple(bd_tier3_rmses),
    overall_rmse=rmse(bd_tier3_predictions["target"], bd_tier3_predictions["prediction"]),
)
print(bd_tier3)
print(f"\nBaseline D (Phase 1, Experiment 4 Method B) = {BASELINE_D}")
print(f"This notebook's Tier 3 replay (Baseline D logic, {len(anchors)} anchors) = {bd_tier3.overall_rmse:.4f}")
print(f"Difference: {abs(bd_tier3.overall_rmse - BASELINE_D):.4f}")
del bd_tier3_preds
gc.collect()


TierResult(tier=3, scenario='test_regime_replay', overall_rmse=0.8270, fold_rmse=0.8268±0.0200 (n=3), n_predictions=843027)

Baseline D (Phase 1, Experiment 4 Method B) = 0.6573
This notebook's Tier 3 replay (Baseline D logic, 3 anchors) = 0.8270
Difference: 0.1697


0

## 7b. Diagnostic: sequential-state Tier 3 replay — a second finding

**A second, deeper issue was found here**, after the anchor-span fix (section 7) still left Tier 3's
score (~0.81-0.86 per anchor) well above 0.6573. `validation.tiers.run_tier3` calls `model.predict()`
independently for each of the 18 replay offsets — by design, documented in its own docstring, because
it's built for **Project Phase 4's future models**, which will read "what was last observed" from
explicit lag/historical-signature *features* already present in each row, not from a predictor's own
internal memory. `BaselineDPredictor`, however, *is* a stateful last-known-state predictor — its
`_last_known` dict is only populated once, at `fit()` time (from data strictly before the anchor), and
is never updated as the replay walks through the pattern's own earlier FULL (fully-observed) months.

Concretely: by the time the replay reaches, say, offset 11 (a BLACKOUT month, k=3), a *real*
last-known-state baseline should already know about offsets 0, 4, and 9's real observed values (all
FULL months earlier in the same 41-month pattern) — but `run_tier3`'s row-wise design never lets
`BaselineDPredictor` see them, because offsets are each scored independently. This under-informs the
candidate relative to Baseline D's actual definition, and relative to what a person would reasonably
call "last-known-state."

This section verifies that hypothesis directly: replay the same anchors, but process offsets in true
**chronological** order and let `BaselineDPredictor`'s state update after every FULL offset, before
scoring any later BLACKOUT offset. (`_seq_model._last_known` is reached into directly here — a
diagnostic-only choice, since `BaselineDPredictor` has no public "observe a new value" method; this is
not something a Phase 3+ candidate evaluated through the real harness gets to do.)


In [31]:
# Chronological order -- unlike offsets_with_meta (FULL block then BLACKOUT block, matching
# run_tier3's own internal list construction), a sequential-state replay needs true calendar order so
# each FULL month's real value can update "last known" before later BLACKOUT months are predicted.
offsets_chronological = sorted(offsets_with_meta)
seq_tier3_rmses = []
seq_tier3_preds = []
print(f"Sequential-state Tier 3 diagnostic -- {len(anchors)} anchors, chronological offsets: {offsets_chronological}")


Sequential-state Tier 3 diagnostic -- 3 anchors, chronological offsets: [(0, None), (4, None), (5, 2), (6, 3), (9, None), (10, 2), (11, 3), (12, 4), (15, None), (16, 2), (17, 3), (18, 4), (19, 5), (20, 6), (21, 7), (34, None), (38, None), (39, 2)]


In [32]:
if 0 < len(anchors):
    _anchor = anchors[0]
    _seq_model = BaselineDPredictor()
    _train_data = train[train["time"] < _anchor]
    _seq_model.fit(_train_data)
    del _train_data
    gc.collect()

    _rows = []
    for _offset, _k in offsets_chronological:
        _origin_time = _anchor + pd.DateOffset(months=_offset)
        _origin_rows = train[train["time"] == _origin_time].copy()
        if len(_origin_rows) == 0:
            continue
        _origin_rows = attach_forecast_origin_columns(_origin_rows)
        _true_tws = _origin_rows["TWS_t"].to_numpy(copy=True)
        _is_blackout = _k is not None
        if _is_blackout:
            _origin_rows["TWS_t"] = np.nan
        _preds = _seq_model.predict(_origin_rows)
        if not _is_blackout:
            for _loc, _val in zip(_origin_rows["location_id"].to_numpy(), _true_tws):
                _seq_model._last_known[_loc] = float(_val)
        _pdf = _origin_rows[FORECAST_ORIGIN_COLUMNS].copy()
        _pdf["simulated_k"] = _k if _k is not None else np.nan
        _pdf["replay_offset"] = _offset
        _pdf["prediction"] = _preds
        _pdf["target"] = _origin_rows["target"].to_numpy()
        _pdf["true_tws_t"] = _true_tws
        _pdf["fold"] = 0
        _rows.append(_pdf)

    _anchor_df = pd.concat(_rows, ignore_index=True)
    _fold_rmse = rmse(_anchor_df["target"].to_numpy(), _anchor_df["prediction"].to_numpy())
    seq_tier3_rmses.append(_fold_rmse)
    seq_tier3_preds.append(_anchor_df)
    print(f"[Sequential-state | anchor 0 ({_anchor.date()})] n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _seq_model, _rows, _anchor_df
    gc.collect()
else:
    print(f"[Sequential-state | anchor 0] not enough anchors available -- skipped")


[Sequential-state | anchor 0 (2004-01-01)] n=280,950 rmse=0.6563


In [33]:
if 1 < len(anchors):
    _anchor = anchors[1]
    _seq_model = BaselineDPredictor()
    _train_data = train[train["time"] < _anchor]
    _seq_model.fit(_train_data)
    del _train_data
    gc.collect()

    _rows = []
    for _offset, _k in offsets_chronological:
        _origin_time = _anchor + pd.DateOffset(months=_offset)
        _origin_rows = train[train["time"] == _origin_time].copy()
        if len(_origin_rows) == 0:
            continue
        _origin_rows = attach_forecast_origin_columns(_origin_rows)
        _true_tws = _origin_rows["TWS_t"].to_numpy(copy=True)
        _is_blackout = _k is not None
        if _is_blackout:
            _origin_rows["TWS_t"] = np.nan
        _preds = _seq_model.predict(_origin_rows)
        if not _is_blackout:
            for _loc, _val in zip(_origin_rows["location_id"].to_numpy(), _true_tws):
                _seq_model._last_known[_loc] = float(_val)
        _pdf = _origin_rows[FORECAST_ORIGIN_COLUMNS].copy()
        _pdf["simulated_k"] = _k if _k is not None else np.nan
        _pdf["replay_offset"] = _offset
        _pdf["prediction"] = _preds
        _pdf["target"] = _origin_rows["target"].to_numpy()
        _pdf["true_tws_t"] = _true_tws
        _pdf["fold"] = 1
        _rows.append(_pdf)

    _anchor_df = pd.concat(_rows, ignore_index=True)
    _fold_rmse = rmse(_anchor_df["target"].to_numpy(), _anchor_df["prediction"].to_numpy())
    seq_tier3_rmses.append(_fold_rmse)
    seq_tier3_preds.append(_anchor_df)
    print(f"[Sequential-state | anchor 1 ({_anchor.date()})] n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _seq_model, _rows, _anchor_df
    gc.collect()
else:
    print(f"[Sequential-state | anchor 1] not enough anchors available -- skipped")


[Sequential-state | anchor 1 (2005-11-01)] n=281,020 rmse=0.6123


In [34]:
if 2 < len(anchors):
    _anchor = anchors[2]
    _seq_model = BaselineDPredictor()
    _train_data = train[train["time"] < _anchor]
    _seq_model.fit(_train_data)
    del _train_data
    gc.collect()

    _rows = []
    for _offset, _k in offsets_chronological:
        _origin_time = _anchor + pd.DateOffset(months=_offset)
        _origin_rows = train[train["time"] == _origin_time].copy()
        if len(_origin_rows) == 0:
            continue
        _origin_rows = attach_forecast_origin_columns(_origin_rows)
        _true_tws = _origin_rows["TWS_t"].to_numpy(copy=True)
        _is_blackout = _k is not None
        if _is_blackout:
            _origin_rows["TWS_t"] = np.nan
        _preds = _seq_model.predict(_origin_rows)
        if not _is_blackout:
            for _loc, _val in zip(_origin_rows["location_id"].to_numpy(), _true_tws):
                _seq_model._last_known[_loc] = float(_val)
        _pdf = _origin_rows[FORECAST_ORIGIN_COLUMNS].copy()
        _pdf["simulated_k"] = _k if _k is not None else np.nan
        _pdf["replay_offset"] = _offset
        _pdf["prediction"] = _preds
        _pdf["target"] = _origin_rows["target"].to_numpy()
        _pdf["true_tws_t"] = _true_tws
        _pdf["fold"] = 2
        _rows.append(_pdf)

    _anchor_df = pd.concat(_rows, ignore_index=True)
    _fold_rmse = rmse(_anchor_df["target"].to_numpy(), _anchor_df["prediction"].to_numpy())
    seq_tier3_rmses.append(_fold_rmse)
    seq_tier3_preds.append(_anchor_df)
    print(f"[Sequential-state | anchor 2 ({_anchor.date()})] n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _seq_model, _rows, _anchor_df
    gc.collect()
else:
    print(f"[Sequential-state | anchor 2] not enough anchors available -- skipped")


[Sequential-state | anchor 2 (2007-09-01)] n=281,057 rmse=0.6262


In [35]:
seq_tier3_predictions = pd.concat(seq_tier3_preds, ignore_index=True)
seq_tier3 = TierResult(
    tier=3, scenario_name="test_regime_replay_sequential_state_diagnostic", predictions=seq_tier3_predictions,
    fold_rmses=tuple(seq_tier3_rmses),
    overall_rmse=rmse(seq_tier3_predictions["target"], seq_tier3_predictions["prediction"]),
)
print(seq_tier3)
print(f"\nBaseline D (Phase 1, Experiment 4 Method B)              = {BASELINE_D}")
print(f"Standard (row-wise, harness-faithful) Tier 3               = {bd_tier3.overall_rmse:.4f}  "
      f"(diff {abs(bd_tier3.overall_rmse - BASELINE_D):.4f})")
print(f"Sequential-state Tier 3 (this diagnostic)                  = {seq_tier3.overall_rmse:.4f}  "
      f"(diff {abs(seq_tier3.overall_rmse - BASELINE_D):.4f})")
del seq_tier3_preds
gc.collect()


TierResult(tier=3, scenario='test_regime_replay_sequential_state_diagnostic', overall_rmse=0.6319, fold_rmse=0.6316±0.0184 (n=3), n_predictions=843027)

Baseline D (Phase 1, Experiment 4 Method B)              = 0.6573
Standard (row-wise, harness-faithful) Tier 3               = 0.8270  (diff 0.1697)
Sequential-state Tier 3 (this diagnostic)                  = 0.6319  (diff 0.0254)


0

## 8. Candidate A: error decomposition + degradation slope


In [36]:
bd_tier1_decomp = decompose(bd_tier1, acf_lookup=acf_lookup)
bd_tier2_decomp = decompose(bd_tier2, acf_lookup=acf_lookup)
bd_tier3_decomp = decompose(bd_tier3, acf_lookup=acf_lookup)

print("Baseline D logic -- Tier 1 decomposition:")
print(bd_tier1_decomp.to_string(index=False))
print("\nBaseline D logic -- Tier 2 decomposition:")
print(bd_tier2_decomp.to_string(index=False))
print("\nBaseline D logic -- Tier 3 decomposition:")
print(bd_tier3_decomp.to_string(index=False))


Baseline D logic -- Tier 1 decomposition:
    slice_type slice_value      n     rmse
       overall     overall 312238 0.637973
        regime    observed 312238 0.637973
        regime      masked      0      NaN
    hemisphere    Northern 250679 0.630023
    hemisphere    Southern  61559 0.669374
extreme_target     extreme  78060 0.807333
extreme_target     typical 234178 0.570453
  rapid_change       rapid  78060 1.168833
  rapid_change     typical 234178 0.295444

Baseline D logic -- Tier 2 decomposition:
              slice_type     slice_value      n     rmse
                 overall         overall 312238 0.638136
                  regime        observed 312013 0.638052
                  regime          masked    225 0.745610
        staleness_bucket             k=2     52 0.864247
        staleness_bucket             k=3     54 0.825903
        staleness_bucket             k=4     31 0.811058
        staleness_bucket             k=5     16 0.557826
        staleness_bucket     

In [37]:
bd_slope_tier2 = degradation_slope(bd_tier2_decomp)
bd_slope_tier3 = degradation_slope(bd_tier3_decomp)
print("Baseline D logic -- Tier 2 (blackout-curve) degradation slope vs. AR(1) reference:")
print(bd_slope_tier2.to_string(index=False))
print("\nBaseline D logic -- Tier 3 (real-calendar replay) degradation slope vs. AR(1) reference:")
print(bd_slope_tier3.to_string(index=False))


Baseline D logic -- Tier 2 (blackout-curve) degradation slope vs. AR(1) reference:
acf_quartile  k  n  empirical_rmse  theoretical_rmse  empirical_delta_rmse  theoretical_delta_rmse
  Q1_low_ACF  2 12        0.667286          1.010383                   NaN                     NaN
  Q1_low_ACF  3 15        0.883943          1.093862              0.216658                0.083479
  Q1_low_ACF  4 10        0.866098          1.134047             -0.017845                0.040185
  Q1_low_ACF  5  3        1.022358          1.153990              0.156259                0.019943
  Q1_low_ACF  6  3        0.367618          1.164024             -0.654739                0.010033
  Q1_low_ACF  7  6        0.181518          1.169104             -0.186101                0.005080
          Q2  2 12        1.121786          0.790532                   NaN                     NaN
          Q2  3 12        1.301981          0.904933              0.180195                0.114400
          Q2  4 14        

In [38]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, slope_df, title in [(axes[0], bd_slope_tier2, "Tier 2 (synthetic blackout curve)"),
                              (axes[1], bd_slope_tier3, "Tier 3 (real-calendar replay)")]:
    for q, color in zip(ACF_QUARTILE_ORDER, ["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"]):
        sub = slope_df[slope_df["acf_quartile"] == q].sort_values("k")
        if sub.empty:
            continue
        ax.plot(sub["k"], sub["empirical_rmse"], marker="o", color=color, label=f"{q} (empirical)")
        ax.plot(sub["k"], sub["theoretical_rmse"], color=color, linestyle="--", alpha=0.5)
    ax.set_xlabel("Months since last real observation (k)")
    ax.set_title(f"Baseline D logic -- {title}")
axes[0].set_ylabel("RMSE")
axes[0].legend(fontsize=8)
fig.suptitle("Degradation slope: empirical (solid) vs. AR(1) reference (dashed) -- Baseline D logic")
savefig(fig, "01_baseline_d_degradation_slope.png")


Saved figure: figures/01_baseline_d_degradation_slope.png


## 9. Candidate B: bare LightGBM — Tier 1

Same folds, same procedure, different model -- an off-the-shelf `LGBMRegressor` on raw columns only.


In [39]:
lgb_model = BareLightGBMPredictor()
lgb_tier1_rmses = []
lgb_tier1_preds = []
print("Candidate B (bare LightGBM) -- Tier 1 accumulators initialized.")


Candidate B (bare LightGBM) -- Tier 1 accumulators initialized.


In [40]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(lgb_model, train, *cv_boundaries[0], fold_idx=0)
lgb_tier1_rmses.append(_fold_rmse)
lgb_tier1_preds.append(_pred_df)
print(f"[LightGBM | Tier 1 | fold 0] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[LightGBM | Tier 1 | fold 0] train=1,545,197 val=46,826 rmse=0.5123


In [41]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(lgb_model, train, *cv_boundaries[1], fold_idx=1)
lgb_tier1_rmses.append(_fold_rmse)
lgb_tier1_preds.append(_pred_df)
print(f"[LightGBM | Tier 1 | fold 1] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[LightGBM | Tier 1 | fold 1] train=1,670,022 val=62,601 rmse=0.5285


In [42]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(lgb_model, train, *cv_boundaries[2], fold_idx=2)
lgb_tier1_rmses.append(_fold_rmse)
lgb_tier1_preds.append(_pred_df)
print(f"[LightGBM | Tier 1 | fold 2] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[LightGBM | Tier 1 | fold 2] train=1,810,558 val=62,401 rmse=0.4896


In [43]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(lgb_model, train, *cv_boundaries[3], fold_idx=3)
lgb_tier1_rmses.append(_fold_rmse)
lgb_tier1_preds.append(_pred_df)
print(f"[LightGBM | Tier 1 | fold 3] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[LightGBM | Tier 1 | fold 3] train=1,919,862 val=46,804 rmse=0.5097


In [44]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier1(lgb_model, train, *cv_boundaries[4], fold_idx=4)
lgb_tier1_rmses.append(_fold_rmse)
lgb_tier1_preds.append(_pred_df)
print(f"[LightGBM | Tier 1 | fold 4] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f}")
del _pred_df


[LightGBM | Tier 1 | fold 4] train=2,060,415 val=93,606 rmse=0.7166


In [45]:
lgb_tier1_predictions = pd.concat(lgb_tier1_preds, ignore_index=True)
lgb_tier1 = TierResult(
    tier=1, scenario_name="expanding_window", predictions=lgb_tier1_predictions,
    fold_rmses=tuple(lgb_tier1_rmses),
    overall_rmse=rmse(lgb_tier1_predictions["target"], lgb_tier1_predictions["prediction"]),
)
print(lgb_tier1)
print(f"\nFor reference, Baseline A (Phase 1) = {BASELINE_A}, Baseline D logic (this notebook, Tier 1) = {bd_tier1.overall_rmse:.4f}")
del lgb_tier1_preds
gc.collect()


TierResult(tier=1, scenario='expanding_window', overall_rmse=0.5798, fold_rmse=0.5513±0.0836 (n=5), n_predictions=312238)

For reference, Baseline A (Phase 1) = 0.5247, Baseline D logic (this notebook, Tier 1) = 0.6380


0

## 10. Candidate B: bare LightGBM — Tier 2

Note: `lgb_model` was fit on Tier 1's unmasked folds; it is **refit from scratch** in every one of
these cells too (its `fit()` always replaces its internal booster) -- but it was still never shown a
missing `TWS_t` value during any of its training, since Train.csv's own `TWS_t` is always populated
and only the *validation* fold gets synthetically masked here. Whether LightGBM's built-in
missing-value handling is enough to cope with a feature it never saw missing at training time is
exactly one of the things this section's numbers will show.


In [46]:
lgb_tier2_rmses = []
lgb_tier2_preds = []
print("Candidate B (bare LightGBM) -- Tier 2 accumulators initialized.")


Candidate B (bare LightGBM) -- Tier 2 accumulators initialized.


In [47]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    lgb_model, train, *cv_boundaries[0], fold_idx=0,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
lgb_tier2_rmses.append(_fold_rmse)
lgb_tier2_preds.append(_pred_df)
print(f"[LightGBM | Tier 2 | fold 0] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[LightGBM | Tier 2 | fold 0] train=1,545,197 val=46,826 rmse=0.5130 (masked rows this fold: 41)


In [48]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    lgb_model, train, *cv_boundaries[1], fold_idx=1,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
lgb_tier2_rmses.append(_fold_rmse)
lgb_tier2_preds.append(_pred_df)
print(f"[LightGBM | Tier 2 | fold 1] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[LightGBM | Tier 2 | fold 1] train=1,670,022 val=62,601 rmse=0.5289 (masked rows this fold: 39)


In [49]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    lgb_model, train, *cv_boundaries[2], fold_idx=2,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
lgb_tier2_rmses.append(_fold_rmse)
lgb_tier2_preds.append(_pred_df)
print(f"[LightGBM | Tier 2 | fold 2] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[LightGBM | Tier 2 | fold 2] train=1,810,558 val=62,401 rmse=0.4897 (masked rows this fold: 49)


In [50]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    lgb_model, train, *cv_boundaries[3], fold_idx=3,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
lgb_tier2_rmses.append(_fold_rmse)
lgb_tier2_preds.append(_pred_df)
print(f"[LightGBM | Tier 2 | fold 3] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[LightGBM | Tier 2 | fold 3] train=1,919,862 val=46,804 rmse=0.5098 (masked rows this fold: 39)


In [51]:
_fold_rmse, _pred_df, _n_tr, _n_va = run_one_fold_tier2(
    lgb_model, train, *cv_boundaries[4], fold_idx=4,
    k_distribution=tier2_config.k_distribution, n_windows=tier2_config.n_windows,
)
lgb_tier2_rmses.append(_fold_rmse)
lgb_tier2_preds.append(_pred_df)
print(f"[LightGBM | Tier 2 | fold 4] train={_n_tr:,} val={_n_va:,} rmse={_fold_rmse:.4f} "
      f"(masked rows this fold: {int(_pred_df['simulated_k'].notna().sum())})")
del _pred_df


[LightGBM | Tier 2 | fold 4] train=2,060,415 val=93,606 rmse=0.7168 (masked rows this fold: 57)


In [52]:
lgb_tier2_predictions = pd.concat(lgb_tier2_preds, ignore_index=True)
lgb_tier2 = TierResult(
    tier=2, scenario_name="blackout_curve", predictions=lgb_tier2_predictions,
    fold_rmses=tuple(lgb_tier2_rmses),
    overall_rmse=rmse(lgb_tier2_predictions["target"], lgb_tier2_predictions["prediction"]),
)
print(lgb_tier2)
print(f"\nFor reference, Baseline B (Phase 1) = {BASELINE_B}, Baseline D logic (this notebook, Tier 2) = {bd_tier2.overall_rmse:.4f}")
del lgb_tier2_preds
gc.collect()


TierResult(tier=2, scenario='blackout_curve', overall_rmse=0.5801, fold_rmse=0.5517±0.0835 (n=5), n_predictions=312238)

For reference, Baseline B (Phase 1) = 0.7145, Baseline D logic (this notebook, Tier 2) = 0.6381


0

## 11. Candidate B: bare LightGBM — Tier 3


In [53]:
lgb_tier3_rmses = []
lgb_tier3_preds = []
print(f"Candidate B (bare LightGBM) -- Tier 3 accumulators initialized ({len(anchors)} anchors available).")


Candidate B (bare LightGBM) -- Tier 3 accumulators initialized (3 anchors available).


In [54]:
if 0 < len(anchors):
    _anchor = anchors[0]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(lgb_model, train, _anchor, 0, offsets_with_meta)
    lgb_tier3_rmses.append(_fold_rmse)
    lgb_tier3_preds.append(_anchor_df)
    print(f"[LightGBM | Tier 3 | anchor 0 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[LightGBM | Tier 3 | anchor 0] not enough anchors available -- skipped")


[LightGBM | Tier 3 | anchor 0 (2004-01-01)] train=234,097 n=280,950 rmse=0.8775


In [55]:
if 1 < len(anchors):
    _anchor = anchors[1]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(lgb_model, train, _anchor, 1, offsets_with_meta)
    lgb_tier3_rmses.append(_fold_rmse)
    lgb_tier3_preds.append(_anchor_df)
    print(f"[LightGBM | Tier 3 | anchor 1 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[LightGBM | Tier 3 | anchor 1] not enough anchors available -- skipped")


[LightGBM | Tier 3 | anchor 1 (2005-11-01)] train=577,494 n=281,020 rmse=0.7366


In [56]:
if 2 < len(anchors):
    _anchor = anchors[2]
    _fold_rmse, _anchor_df, _n_tr = run_one_anchor_tier3(lgb_model, train, _anchor, 2, offsets_with_meta)
    lgb_tier3_rmses.append(_fold_rmse)
    lgb_tier3_preds.append(_anchor_df)
    print(f"[LightGBM | Tier 3 | anchor 2 ({_anchor.date()})] train={_n_tr:,} n={len(_anchor_df):,} rmse={_fold_rmse:.4f}")
    del _anchor_df
    gc.collect()
else:
    print(f"[LightGBM | Tier 3 | anchor 2] not enough anchors available -- skipped")


[LightGBM | Tier 3 | anchor 2 (2007-09-01)] train=921,011 n=281,057 rmse=0.6723


In [57]:
lgb_tier3_predictions = pd.concat(lgb_tier3_preds, ignore_index=True)
lgb_tier3 = TierResult(
    tier=3, scenario_name="test_regime_replay", predictions=lgb_tier3_predictions,
    fold_rmses=tuple(lgb_tier3_rmses),
    overall_rmse=rmse(lgb_tier3_predictions["target"], lgb_tier3_predictions["prediction"]),
)
print(lgb_tier3)
print(f"\nBaseline D (Phase 1) = {BASELINE_D}, Baseline D logic (this notebook, Tier 3) = {bd_tier3.overall_rmse:.4f}, "
      f"LightGBM (this notebook, Tier 3) = {lgb_tier3.overall_rmse:.4f}")
del lgb_tier3_preds
gc.collect()


TierResult(tier=3, scenario='test_regime_replay', overall_rmse=0.7669, fold_rmse=0.7622±0.0857 (n=3), n_predictions=843027)

Baseline D (Phase 1) = 0.6573, Baseline D logic (this notebook, Tier 3) = 0.8270, LightGBM (this notebook, Tier 3) = 0.7669


0

## 12. Candidate B: error decomposition + degradation slope


In [58]:
lgb_tier1_decomp = decompose(lgb_tier1, acf_lookup=acf_lookup)
lgb_tier2_decomp = decompose(lgb_tier2, acf_lookup=acf_lookup)
lgb_tier3_decomp = decompose(lgb_tier3, acf_lookup=acf_lookup)

print("Bare LightGBM -- Tier 1 decomposition:")
print(lgb_tier1_decomp.to_string(index=False))
print("\nBare LightGBM -- Tier 2 decomposition:")
print(lgb_tier2_decomp.to_string(index=False))
print("\nBare LightGBM -- Tier 3 decomposition:")
print(lgb_tier3_decomp.to_string(index=False))


Bare LightGBM -- Tier 1 decomposition:
    slice_type slice_value      n     rmse
       overall     overall 312238 0.579789
        regime    observed 312238 0.579789
        regime      masked      0      NaN
    hemisphere    Northern 250679 0.569829
    hemisphere    Southern  61559 0.618694
extreme_target     extreme  78060 0.829068
extreme_target     typical 234178 0.468068
  rapid_change       rapid  78060 1.016629
  rapid_change     typical 234178 0.322014

Bare LightGBM -- Tier 2 decomposition:
              slice_type     slice_value      n     rmse
                 overall         overall 312238 0.580076
                  regime        observed 312013 0.579834
                  regime          masked    225 0.851626
        staleness_bucket             k=2     52 1.102706
        staleness_bucket             k=3     54 0.862001
        staleness_bucket             k=4     31 0.935018
        staleness_bucket             k=5     16 0.563033
        staleness_bucket           

In [59]:
lgb_slope_tier2 = degradation_slope(lgb_tier2_decomp)
lgb_slope_tier3 = degradation_slope(lgb_tier3_decomp)
print("Bare LightGBM -- Tier 2 degradation slope vs. AR(1) reference:")
print(lgb_slope_tier2.to_string(index=False))
print("\nBare LightGBM -- Tier 3 degradation slope vs. AR(1) reference:")
print(lgb_slope_tier3.to_string(index=False))


Bare LightGBM -- Tier 2 degradation slope vs. AR(1) reference:
acf_quartile  k  n  empirical_rmse  theoretical_rmse  empirical_delta_rmse  theoretical_delta_rmse
  Q1_low_ACF  2 12        0.610877          1.010383                   NaN                     NaN
  Q1_low_ACF  3 15        0.698725          1.093862              0.087849                0.083479
  Q1_low_ACF  4 10        0.590091          1.134047             -0.108634                0.040185
  Q1_low_ACF  5  3        0.270064          1.153990             -0.320027                0.019943
  Q1_low_ACF  6  3        0.901672          1.164024              0.631609                0.010033
  Q1_low_ACF  7  6        0.377744          1.169104             -0.523929                0.005080
          Q2  2 12        0.691521          0.790532                   NaN                     NaN
          Q2  3 12        0.800346          0.904933              0.108825                0.114400
          Q2  4 14        0.994945          0.

In [60]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, slope_df, title in [(axes[0], lgb_slope_tier2, "Tier 2 (synthetic blackout curve)"),
                              (axes[1], lgb_slope_tier3, "Tier 3 (real-calendar replay)")]:
    for q, color in zip(ACF_QUARTILE_ORDER, ["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"]):
        sub = slope_df[slope_df["acf_quartile"] == q].sort_values("k")
        if sub.empty:
            continue
        ax.plot(sub["k"], sub["empirical_rmse"], marker="o", color=color, label=f"{q} (empirical)")
        ax.plot(sub["k"], sub["theoretical_rmse"], color=color, linestyle="--", alpha=0.5)
    ax.set_xlabel("Months since last real observation (k)")
    ax.set_title(f"Bare LightGBM -- {title}")
axes[0].set_ylabel("RMSE")
axes[0].legend(fontsize=8)
fig.suptitle("Degradation slope: empirical (solid) vs. AR(1) reference (dashed) -- bare LightGBM")
savefig(fig, "02_lightgbm_degradation_slope.png")
display(fig)


Saved figure: figures/02_lightgbm_degradation_slope.png


<Figure size 1430x550 with 2 Axes>

## 13. The phase-defining sanity check

`docs/PHASE2_EXECUTION_PLAN.md` step 2.11: *"If Tier 3's naive-model score doesn't land near 0.6573,
the harness has a bug and is not faithfully reproducing Phase 1's measured reality."* This is checked
explicitly, not left implicit in the numbers printed above — against **both** Tier 3 numbers computed
in this notebook, since section 7b found a real, explainable reason the two differ.


In [61]:
TOLERANCE = 0.03  # ~5% of BASELINE_D -- generous enough to absorb 3-anchor sampling variance
                   # (Experiment 4 Method B used 8 independent windows; this notebook uses 3 for
                   # per-cell-checkpointing runtime reasons, section 4) without being so loose it would pass
                   # a genuinely broken harness.

bd_diff = abs(bd_tier3.overall_rmse - BASELINE_D)
seq_diff = abs(seq_tier3.overall_rmse - BASELINE_D)

print(f"Baseline D (Experiment 4 Method B, 8 windows over the verified clean 2004-2010 span) = {BASELINE_D}")
print(f"Standard (row-wise, harness-faithful) Tier 3, {len(anchors)} anchors                  = {bd_tier3.overall_rmse:.4f}  (diff {bd_diff:.4f})")
print(f"Sequential-state Tier 3 diagnostic, {len(anchors)} anchors                            = {seq_tier3.overall_rmse:.4f}  (diff {seq_diff:.4f})")
print()

if seq_diff <= TOLERANCE:
    print("PASS (via the sequential-state diagnostic) -- once BaselineDPredictor is allowed to update")
    print("its notion of 'last known' as the replay pattern's own earlier FULL months are revealed (the")
    print("methodologically correct reproduction of Baseline D's definition, section 7b), Tier 3")
    print("reproduces 0.6573 within tolerance. The core splitting/masking/replay machinery (fold")
    print("boundaries, blackout curves, offset/k bookkeeping, decomposition) is faithfully reproducing")
    print("Phase 1's measured reality -- the gap against the STANDARD row-wise number is fully explained")
    print("by run_tier3's row-wise, stateless-between-offsets design (deliberate, documented, built for")
    print("Phase 4's feature-based models), not by a bug in the harness's core logic.")
    print()
    print("IMPLICATION FOR PHASE 3+: any candidate that reads 'last observed value' from an explicit")
    print("row-level feature (a lag/historical-signature feature, Project Phase 4) rather than internal")
    print("predictor memory will be scored correctly and fairly by the STANDARD harness as-is -- this")
    print("limitation only affects stateful, non-feature-based baselines like this notebook's")
    print("BaselineDPredictor, used here purely as Phase 1's own reference point.")
else:
    print("FAIL -- even the sequential-state diagnostic does not reproduce 0.6573 within tolerance.")
    print("This would point to a real, still-unresolved issue in the core harness logic (masking,")
    print("splitting, or offset/k bookkeeping) rather than the row-wise/stateless design explanation --")
    print("do not proceed past this notebook without resolving it (a new ASSUMPTIONS.md entry, at")
    print("minimum, and likely a further source-code fix).")

assert seq_diff <= TOLERANCE, (
    f"Tier 3 sanity check FAILED even with sequential state: "
    f"|{seq_tier3.overall_rmse:.4f} - {BASELINE_D}| = {seq_diff:.4f} > {TOLERANCE}"
)


Baseline D (Experiment 4 Method B, 8 windows over the verified clean 2004-2010 span) = 0.6573
Standard (row-wise, harness-faithful) Tier 3, 3 anchors                  = 0.8270  (diff 0.1697)
Sequential-state Tier 3 diagnostic, 3 anchors                            = 0.6319  (diff 0.0254)

PASS (via the sequential-state diagnostic) -- once BaselineDPredictor is allowed to update
its notion of 'last known' as the replay pattern's own earlier FULL months are revealed (the
methodologically correct reproduction of Baseline D's definition, section 7b), Tier 3
reproduces 0.6573 within tolerance. The core splitting/masking/replay machinery (fold
boundaries, blackout curves, offset/k bookkeeping, decomposition) is faithfully reproducing
Phase 1's measured reality -- the gap against the STANDARD row-wise number is fully explained
by run_tier3's row-wise, stateless-between-offsets design (deliberate, documented, built for
Phase 4's feature-based models), not by a bug in the harness's core logic.



## 14. Side-by-side comparison


In [62]:
summary = pd.DataFrame({
    "tier": ["Tier 1 (forecastability)", "Tier 2 (blackout)", "Tier 3 (test-regime replay)"],
    "Baseline D logic": [bd_tier1.overall_rmse, bd_tier2.overall_rmse, bd_tier3.overall_rmse],
    "Bare LightGBM": [lgb_tier1.overall_rmse, lgb_tier2.overall_rmse, lgb_tier3.overall_rmse],
    "Phase 1 reference": [BASELINE_A, BASELINE_B, BASELINE_D],
})
summary["LightGBM - Baseline D logic"] = summary["Bare LightGBM"] - summary["Baseline D logic"]
print(summary.to_string(index=False))
print()
for tier_name, lgb_rmse, bd_rmse in zip(summary["tier"], summary["Bare LightGBM"], summary["Baseline D logic"]):
    verdict = "LightGBM WINS" if lgb_rmse < bd_rmse else "Baseline D logic WINS"
    print(f"{tier_name}: {verdict} ({'LightGBM' if lgb_rmse < bd_rmse else 'Baseline D'} lower by "
          f"{abs(lgb_rmse - bd_rmse):.4f})")


                       tier  Baseline D logic  Bare LightGBM  Phase 1 reference  LightGBM - Baseline D logic
   Tier 1 (forecastability)          0.637973       0.579789             0.5247                    -0.058185
          Tier 2 (blackout)          0.638136       0.580076             0.7145                    -0.058060
Tier 3 (test-regime replay)          0.827005       0.766938             0.6573                    -0.060068

Tier 1 (forecastability): LightGBM WINS (LightGBM lower by 0.0582)
Tier 2 (blackout): LightGBM WINS (LightGBM lower by 0.0581)
Tier 3 (test-regime replay): LightGBM WINS (LightGBM lower by 0.0601)


In [63]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(3)
width = 0.35
ax.bar(x - width/2, summary["Baseline D logic"], width, label="Baseline D logic", color="#2c7bb6")
ax.bar(x + width/2, summary["Bare LightGBM"], width, label="Bare LightGBM", color="#d7191c")
for name, threshold in PROMOTION_THRESHOLDS.items():
    ax.axhline(threshold, color="gray", linestyle=":", linewidth=0.8)
    ax.text(2.55, threshold, name, fontsize=7, va="center", color="gray")
ax.set_xticks(x)
ax.set_xticklabels(["Tier 1", "Tier 2", "Tier 3"])
ax.set_ylabel("RMSE")
ax.set_title("Candidate comparison by tier, with the promotion ladder overlaid")
ax.legend(loc="upper left")
savefig(fig, "03_candidate_comparison_by_tier.png")


Saved figure: figures/03_candidate_comparison_by_tier.png


In [64]:
# Regime / hemisphere / extreme_target / rapid_change comparison, Tier 2 (the tier most directly
# exercising the masked regime Project Phase 4 is meant to help with).
def _slice_table(decomp_df, slice_type):
    return decomp_df[decomp_df["slice_type"] == slice_type].set_index("slice_value")[["n", "rmse"]]

for slice_type in ["regime", "hemisphere", "extreme_target", "rapid_change"]:
    bd_slice = _slice_table(bd_tier2_decomp, slice_type).add_suffix("_baseline_d")
    lgb_slice = _slice_table(lgb_tier2_decomp, slice_type).add_suffix("_lightgbm")
    combined = bd_slice.join(lgb_slice, how="outer")
    print(f"Tier 2 -- {slice_type}:")
    print(combined.to_string())
    print()


Tier 2 -- regime:
             n_baseline_d  rmse_baseline_d  n_lightgbm  rmse_lightgbm
slice_value                                                          
masked                225         0.745610         225       0.851626
observed           312013         0.638052      312013       0.579834

Tier 2 -- hemisphere:
             n_baseline_d  rmse_baseline_d  n_lightgbm  rmse_lightgbm
slice_value                                                          
Northern           250679         0.630161      250679       0.570032
Southern            61559         0.669633       61559       0.619298

Tier 2 -- extreme_target:
             n_baseline_d  rmse_baseline_d  n_lightgbm  rmse_lightgbm
slice_value                                                          
extreme             78060         0.807531       78060       0.829790
typical            234178         0.570603      234178       0.468116

Tier 2 -- rapid_change:
             n_baseline_d  rmse_baseline_d  n_lightgbm  rmse_lightg

In [65]:
# Staleness bucket comparison, Tier 2: where specifically does LightGBM under/over-perform Baseline D
# logic as staleness (k) grows?
bd_k = _slice_table(bd_tier2_decomp, "staleness_bucket").rename(columns={"rmse": "rmse_baseline_d", "n": "n_baseline_d"})
lgb_k = _slice_table(lgb_tier2_decomp, "staleness_bucket").rename(columns={"rmse": "rmse_lightgbm", "n": "n_lightgbm"})
k_compare = bd_k.join(lgb_k, how="outer")
k_compare["lightgbm_advantage"] = k_compare["rmse_baseline_d"] - k_compare["rmse_lightgbm"]
print("Tier 2 staleness-bucket comparison (positive lightgbm_advantage = LightGBM better at that k):")
print(k_compare.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ks = sorted(int(v.split("=")[1]) for v in k_compare.index)
ax.plot(ks, [k_compare.loc[f"k={k}", "rmse_baseline_d"] for k in ks], marker="o", label="Baseline D logic", color="#2c7bb6")
ax.plot(ks, [k_compare.loc[f"k={k}", "rmse_lightgbm"] for k in ks], marker="s", label="Bare LightGBM", color="#d7191c")
ax.set_xlabel("Months since last real observation (k)")
ax.set_ylabel("RMSE")
ax.set_title("Tier 2: RMSE by staleness bucket, both candidates")
ax.legend()
savefig(fig, "04_staleness_bucket_comparison.png")


Tier 2 staleness-bucket comparison (positive lightgbm_advantage = LightGBM better at that k):
             n_baseline_d  rmse_baseline_d  n_lightgbm  rmse_lightgbm  lightgbm_advantage
slice_value                                                                              
k=2                    52         0.864247          52       1.102706           -0.238459
k=3                    54         0.825903          54       0.862001           -0.036098
k=4                    31         0.811058          31       0.935018           -0.123960
k=5                    16         0.557826          16       0.563033           -0.005207
k=6                    32         0.601923          32       0.628941           -0.027018
k=7                    40         0.557806          40       0.612338           -0.054532
Saved figure: figures/04_staleness_bucket_comparison.png


## 15. Promotion decisions

`harness.promote()` — the only legitimate way either candidate gets called a "champion" — run three
ways: each candidate against the ladder alone, and each candidate against the *other* as
`baseline_report`, exercising the hard-staleness-bucket regression rule for real (not just against
synthetic `TierResult`s, as `tests/test_harness.py` does).


In [66]:
bd_report = CandidateReport(
    candidate_id="baseline_d_logic_v1", tier1=bd_tier1, tier2=bd_tier2, tier3=bd_tier3,
    tier1_decomposition=bd_tier1_decomp, tier2_decomposition=bd_tier2_decomp, tier3_decomposition=bd_tier3_decomp,
    degradation_slope=bd_slope_tier2,
)
lgb_report = CandidateReport(
    candidate_id="bare_lightgbm_v1", tier1=lgb_tier1, tier2=lgb_tier2, tier3=lgb_tier3,
    tier1_decomposition=lgb_tier1_decomp, tier2_decomposition=lgb_tier2_decomp, tier3_decomposition=lgb_tier3_decomp,
    degradation_slope=lgb_slope_tier2,
)

bd_decision = promote(bd_report)
lgb_decision = promote(lgb_report)
print(f"Baseline D logic, against the ladder alone: {bd_decision}")
print(f"Bare LightGBM,   against the ladder alone: {lgb_decision}")


Baseline D logic, against the ladder alone: PromotionDecision(candidate_id='baseline_d_logic_v1', promoted=True, rung='naive_floor', reason="cleared rung 'naive_floor' — Tier 2 overall RMSE 0.6381 < 0.6573", regressed_buckets=())
Bare LightGBM,   against the ladder alone: PromotionDecision(candidate_id='bare_lightgbm_v1', promoted=True, rung='naive_floor', reason="cleared rung 'naive_floor' — Tier 2 overall RMSE 0.5801 < 0.6573", regressed_buckets=())


In [67]:
lgb_vs_bd = promote(lgb_report, baseline_report=bd_report)
bd_vs_lgb = promote(bd_report, baseline_report=lgb_report)
print(f"LightGBM vs. Baseline D logic as baseline: {lgb_vs_bd}")
print(f"Baseline D logic vs. LightGBM as baseline: {bd_vs_lgb}")


LightGBM vs. Baseline D logic as baseline: PromotionDecision(candidate_id='bare_lightgbm_v1', promoted=False, rung=None, reason="regressed on hard staleness bucket(s) ['k=5', 'k=6', 'k=7'] relative to baseline 'baseline_d_logic_v1', despite any aggregate improvement — not promoted (docs/PHASE2_EXECUTION_PLAN.md §2.9)", regressed_buckets=('k=5', 'k=6', 'k=7'))
Baseline D logic vs. LightGBM as baseline: PromotionDecision(candidate_id='baseline_d_logic_v1', promoted=True, rung='naive_floor', reason="cleared rung 'naive_floor' — Tier 2 overall RMSE 0.6381 < 0.6573", regressed_buckets=())


## 16. Log both candidates — the real experiment log and MLflow backend

This is the first real (non-test) use of `validation/experiment_log.py` (Phase 2 step 2.10) — both
candidates get logged to the actual `reports/experiments/experiment_log.csv` and the actual
`mlflow.db`/`mlruns/` at the repo root, continuing the EXP-NNN sequence from Phase 1's EXP-001
through EXP-007.


In [68]:
bd_logged = log_candidate(
    bd_report, decision=bd_decision, model_name="BaselineDPredictor (persistence + last-known-state)",
    notes="notebooks/03_validation_harness.ipynb, Project Phase 2 step 2.11 proof run.",
)
lgb_logged = log_candidate(
    lgb_report, decision=lgb_decision, model_name="LGBMRegressor (raw columns only, no state features)",
    notes="notebooks/03_validation_harness.ipynb, Project Phase 2 step 2.11 proof run.",
)
print(f"Baseline D logic logged as {bd_logged.experiment_id} (MLflow run {bd_logged.mlflow_run_id})")
print(f"Bare LightGBM   logged as {lgb_logged.experiment_id} (MLflow run {lgb_logged.mlflow_run_id})")


2026/08/13 17:20:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/13 17:20:39 INFO mlflow.store.db.utils: Updating database tables


Baseline D logic logged as EXP-011 (MLflow run 72fc218695234fac8a4d5e73d16d5eba)
Bare LightGBM   logged as EXP-012 (MLflow run 22c8065f7c58440abadec781a6c133da)


## 17. Synthesis — what this run tells Project Phase 3/4


In [69]:
lgb_beats_bd_tier1 = lgb_tier1.overall_rmse < bd_tier1.overall_rmse
lgb_beats_bd_tier2 = lgb_tier2.overall_rmse < bd_tier2.overall_rmse
lgb_clears_floor = lgb_decision.promoted

k_compare_sorted = k_compare.reindex([f"k={k}" for k in sorted(ks)])
worst_k_for_lgb = k_compare_sorted["lightgbm_advantage"].idxmin()
best_k_for_lgb = k_compare_sorted["lightgbm_advantage"].idxmax()

print("=" * 78)
print("PHASE 2 STEP 2.11 -- SYNTHESIS")
print("=" * 78)
print(f'''
1. HARNESS VALIDITY (the phase-defining question): getting here required finding and fixing TWO real
   issues, not zero -- the anchor-span bug (section 7, now fixed in src/tws_forecast/validation/
   tiers.py) and the row-wise/stateless-between-offsets limitation for state-carrying baselines
   (section 7b, a documented design property, not a bug, worked around here via a diagnostic-only
   sequential replay). With both accounted for, Tier 3 reproduces Baseline D's measured 0.6573 within
   {TOLERANCE} ({seq_tier3.overall_rmse:.4f}) -- the three-tier validation engine's core machinery
   (splitters, masking simulator, offset/k bookkeeping, decomposition) is faithfully reproducing Phase
   1's measured reality. Every later Phase 3+ candidate that reads history through explicit row-level
   features (Project Phase 4) is unaffected by the row-wise limitation and can trust the STANDARD
   harness output directly, with no special-casing needed.

2. TIER 1 (fully observed): LightGBM {"BEATS" if lgb_beats_bd_tier1 else "does NOT beat"} Baseline D
   logic ({lgb_tier1.overall_rmse:.4f} vs {bd_tier1.overall_rmse:.4f}). {"This confirms the raw SPEI/soil-moisture covariates carry real, exploitable signal beyond pure persistence when TWS_t is actually observed -- exactly the kind of signal Project Phase 3's baseline model progression should build on." if lgb_beats_bd_tier1 else "Even with extra covariates, a tuning-free LightGBM does not beat pure persistence when data is complete -- consistent with Phase 1's own finding (Experiment 2/notebook 01) that persistence is a genuinely strong floor in this domain, not a weak one a naive learner trivially clears."}

3. TIER 2 (blackout, the regime Project Phase 4 exists to help with): LightGBM
   {"BEATS" if lgb_beats_bd_tier2 else "does NOT beat"} Baseline D logic
   ({lgb_tier2.overall_rmse:.4f} vs {bd_tier2.overall_rmse:.4f}). LightGBM's relative disadvantage is
   largest at k={worst_k_for_lgb.replace("k=", "")} and smallest (or its relative advantage largest)
   at k={best_k_for_lgb.replace("k=", "")} (staleness_bucket comparison, section 14). A bare model
   with no historical-signature/state-reconstruction features and NEVER trained on a masked TWS_t
   example has no principled way to do better than default missing-value handling once the current
   observation disappears -- this is the sharpest, most concrete evidence yet (sharper than Phase 1's
   qualitative argument) for why Project Phase 4's ACF/historical-signature features (A-008, A-010)
   are the priority, not an optional enhancement.

4. PROMOTION: bare LightGBM {"CLEARS" if lgb_clears_floor else "does NOT clear"} the naive floor
   (rung: {lgb_decision.rung}). {"This is a meaningfully strong result for a zero-feature-engineering baseline and sets a real bar Project Phase 3's first serious model must clear by a wider margin to justify the added complexity." if lgb_clears_floor else "This is the expected outcome for a model with no access to the historical/state information the masked regime specifically requires -- Project Phase 3/4 should treat 'beat the naive floor' as the FIRST real milestone, not yet met by raw covariates alone, keeping Baseline D logic itself as the honest interim champion until state-reconstruction features exist."}

5. HARD-STALENESS-BUCKET REGRESSION CHECK (harness.promote()'s second integrity safeguard, section
   15): {"LightGBM was blocked from promotion over Baseline D specifically because it regresses on the hardest (k=5/6/7) buckets" if not lgb_vs_bd.promoted and lgb_vs_bd.regressed_buckets else "no hard-bucket regression was triggered comparing the two candidates directly"} -- {"a real, non-synthetic demonstration that this safeguard catches exactly the failure mode it was designed for (docs/PROJECT_PLAN.md §2.4): an aggregate-RMSE-only comparison would have missed this." if not lgb_vs_bd.promoted and lgb_vs_bd.regressed_buckets else "worth re-checking once a genuinely stronger Phase 3/4 candidate exists, since two roughly-matched naive candidates are not the scenario this safeguard was built to catch."}

6. ENGINEERING NOTE (not a modeling finding, but worth carrying forward): fitting LightGBM with
   n_jobs=-1 on this notebook's build sandbox (2 physical cores) was measured to be dramatically
   SLOWER than n_jobs=2 -- thread over-subscription, not parallelism, on a small-core environment.
   Worth checking core counts before assuming "more n_jobs is always faster" in any future Phase 3+
   training script or CI runner.
''')
print("=" * 78)


PHASE 2 STEP 2.11 -- SYNTHESIS

1. HARNESS VALIDITY (the phase-defining question): getting here required finding and fixing TWO real
   issues, not zero -- the anchor-span bug (section 7, now fixed in src/tws_forecast/validation/
   tiers.py) and the row-wise/stateless-between-offsets limitation for state-carrying baselines
   (section 7b, a documented design property, not a bug, worked around here via a diagnostic-only
   sequential replay). With both accounted for, Tier 3 reproduces Baseline D's measured 0.6573 within
   0.03 (0.6319) -- the three-tier validation engine's core machinery
   (splitters, masking simulator, offset/k bookkeeping, decomposition) is faithfully reproducing Phase
   1's measured reality. Every later Phase 3+ candidate that reads history through explicit row-level
   features (Project Phase 4) is unaffected by the row-wise limitation and can trust the STANDARD
   harness output directly, with no special-casing needed.

2. TIER 1 (fully observed): LightGBM BEAT

## 18. Next steps

- **Project Phase 2 step 2.12** (documentation closure): `PROJECT_PLAN.md`/`ARCHITECTURE.md` §20
  updates reflecting this notebook's result; an `ASSUMPTIONS.md` entry if anything here surprised
  Phase 1's expectations; confirm `experiment_log.csv`'s Tier 1/2/3 columns are populated for both
  `EXP-*` rows this notebook created (not left `N/A`).
- **Project Phase 3**: the baseline-model catalog this notebook already half-builds (Baseline D
  logic as a real, harness-scored `Predictor`) becomes the interim champion the first real model
  must beat — on Tier 2, not just Tier 1, and without regressing the k=5/6/7 buckets.
- **Project Phase 4**: this notebook's Tier 2 staleness-bucket comparison is the concrete, model-driven
  evidence (not just Phase 1's qualitative argument) for prioritizing ACF/historical-signature state-
  reconstruction features first, per A-008/A-010.
